In [1]:
import sys
import pandas as pd
from pathlib import Path

sys.path.append('/Users/BKieft/Metabolomics/metatlas2/metatlas2')
import workflow_objects as wfo
import load_tools as ldt
import logging_config as lcf
import database_interact as dbi

logger = lcf.get_logger('workflow_example')

pd.options.display.max_colwidth = 300

In [2]:
# Control flags
new_main_database = False
new_atlases = False
new_analysis = True

# Load configuration
ANALYSIS_CONFIG = ldt.load_metatlas2_config("/Users/BKieft/Metabolomics/metatlas2/configs/analysis.yaml")
ATLAS_CONFIG = ldt.load_atlas_config("/Users/BKieft/Metabolomics/metatlas2/configs/create_atlases.yaml")
COMPOUNDS_CONFIG = ldt.load_compound_config("/Users/BKieft/Metabolomics/metatlas2/configs/create_compounds.yaml")

2025-09-16 15:34:30 | metatlas2.load_tools | INFO | Loaded metatlas2 configuration from /Users/BKieft/Metabolomics/metatlas2/configs/analysis.yaml


In [3]:
if new_main_database:
    db_manager = wfo.DatabaseManager(COMPOUNDS_CONFIG, overwrite_db=True)
    compounds, compound_references = db_manager.create_compound_db_entries()
    
    logger.info(f"Main database created!")
    logger.info(f"  - Compounds created: {len(compounds)}")
    logger.info(f"  - Compound references created: {len(compound_references)}")
    
else:
    logger.info("Skipping main database creation (new_main_database=False)")

2025-09-16 15:34:30 | metatlas2.workflow_example | INFO | Skipping main database creation (new_main_database=False)


In [4]:
if new_atlases:
    atlas_manager = wfo.AtlasManager(ATLAS_CONFIG)
    created_atlases = atlas_manager.create_atlas_from_file()
    
    logger.info("Atlases created successfully!")
    logger.info(f"Total atlases created: {len(created_atlases)}")
    
    for atlas in created_atlases:
        logger.info(f"  Atlas: {atlas.atlas_name}")
        logger.info(f"    UID: {atlas.atlas_uid}")
        logger.info(f"    Method: {atlas.chromatography}/{atlas.polarity}")
        logger.info(f"    Compounds: {len(atlas.compound_references)}")
            
else:
    logger.info("Using existing atlas UIDs from config")

2025-09-16 15:34:30 | metatlas2.workflow_example | INFO | Using existing atlas UIDs from config


In [5]:
if new_analysis:

    # Setup project paths
    PROJECT_NAME = "20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519"
    PROJECT_DIRECTORY = f"{ANALYSIS_CONFIG['ENV']['PATHS']['projects_dir']}/{PROJECT_NAME}"
    PROJECT_DB_PATH = f"{PROJECT_DIRECTORY}/{PROJECT_NAME}.duckdb"
    PROJECT_FILES_PATH = f"{ANALYSIS_CONFIG['ENV']['PATHS']['projects_dir']}/{PROJECT_NAME}/lcmsruns"

    # Ensure project directory exists
    Path(PROJECT_DIRECTORY).mkdir(parents=True, exist_ok=True)

    # Create TargetedAnalysisManager with proper variable name
    analysis_manager = wfo.TargetedAnalysisManager(
        config=ANALYSIS_CONFIG,
        project_directory=PROJECT_DIRECTORY,
        project_name=PROJECT_NAME,
        project_lcmsruns_path=PROJECT_FILES_PATH,
        rt_alignment_num=2,
        analysis_num=0
    )

    # Run the complete workflow
    analysis_manager.run_complete_workflow(
        stop_at_stage="putative_identification",  # Options: data_extraction, rt_alignment, putative_identification, manual_curation
        analysis_subset=[('hilicz', 'istd', 'pos'), ('hilicz', 'qc', 'pos')],  # Use lowercase as expected
        create_analysis_notebooks=True 
    )
    
    logger.info("Analysis workflow completed!")

2025-09-16 15:34:30 | metatlas2.workflow_objects | INFO | Processing workflow: HILICZ
2025-09-16 15:34:30 | metatlas2.workflow_objects | INFO | RT alignment atlas: atl-raw-qc-bd654f5e1d4d40c493aef321a962933f (workflow: HILICZ)
2025-09-16 15:34:30 | metatlas2.workflow_objects | INFO | Analysis atlas: HILICZ/QC/POS -> atl-raw-qc-bd654f5e1d4d40c493aef321a962933f
2025-09-16 15:34:30 | metatlas2.workflow_objects | INFO | No atlas UID provided for HILICZ/QC/NEG - skipping
2025-09-16 15:34:30 | metatlas2.workflow_objects | INFO | Analysis atlas: HILICZ/ISTD/POS -> atl-raw-istd-2de05155672c4160b39d71bca895a85c
2025-09-16 15:34:30 | metatlas2.workflow_objects | INFO | No atlas UID provided for HILICZ/ISTD/NEG - skipping
2025-09-16 15:34:30 | metatlas2.workflow_objects | INFO | Analysis atlas: HILICZ/EMA/POS -> atl-raw-ema-897c5e87af094afa98f99bb92c77f25a
2025-09-16 15:34:30 | metatlas2.workflow_objects | INFO | No atlas UID provided for HILICZ/EMA/NEG - skipping
2025-09-16 15:34:30 | metatlas2.

Categorizing files:   0%|          | 0/8 [00:00<?, ?it/s]

2025-09-16 15:34:30 | metatlas2.database_interact | WARNING | LCMS runs already exist in the database. Proceeding with these files.
2025-09-16 15:34:30 | metatlas2.database_interact | INFO | Chromatography: HILIC
2025-09-16 15:34:30 | metatlas2.database_interact | INFO |   Polarity: FPS
2025-09-16 15:34:30 | metatlas2.database_interact | INFO |     qc: 2 files
2025-09-16 15:34:30 | metatlas2.database_interact | INFO |     istd: 1 files
2025-09-16 15:34:30 | metatlas2.database_interact | INFO |     experimental: 1 files
2025-09-16 15:34:30 | metatlas2.database_interact | INFO |   Polarity: positive
2025-09-16 15:34:30 | metatlas2.database_interact | INFO |     experimental: 3 files
2025-09-16 15:34:30 | metatlas2.database_interact | INFO |     exctrl: 1 files
2025-09-16 15:34:30 | metatlas2.workflow_objects | INFO | LCMS runs loaded successfully
2025-09-16 15:34:31 | metatlas2.workflow_objects | INFO | ========== Stage 2: RT Correction ==========
2025-09-16 15:34:31 | metatlas2.workflow

Processing files in parallel:   0%|          | 0/6 [00:00<?, ?it/s]

2025-09-16 15:34:34 | metatlas2.ms1_ms2_analysis | INFO |   Completed 1/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_22_Fall-RFa_7_Rg70to1050-CE205060norm-rhizo-InjBL-MeOH_Run123.h5 - 12 compounds
2025-09-16 15:34:34 | metatlas2.ms1_ms2_analysis | INFO |   Completed 2/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_28_Spring-VSp_2_Rg70to1050-CE102040norm-veg-core-ISTD_Run31.h5 - 20 compounds
2025-09-16 15:34:35 | metatlas2.ms1_ms2_analysis | INFO |   Completed 3/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_36_Fall-VFa_7_Rg70to1050-CE102040norm-veg-core-S1_Run136.h5 - 19 compounds
2025-09-16 15:34:35 | metatlas2.ms1_ms2_analysis | INFO |   Completed 4/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_48_Fall-BFa_7_Rg70to1050-CE102040norm-nonveg-core-S1_Run127.h5 - 20 compounds
2025-09-16 15:34:35 | metatlas2.ms1_ms2_analysis | INFO |   Completed 5/6: 20250707_JGI_MS_510172

...

2025-09-16 15:34:41 | metatlas2.ms1_ms2_analysis | INFO |   Hit detection 1/14 complete: OVRNDRQMDRJTHS-ZEUBEQSHSA-N
2025-09-16 15:34:41 | metatlas2.ms1_ms2_analysis | INFO |   Hit detection 2/14 complete: GFFGJBXGBJISGV-CIKZIQIKSA-N
2025-09-16 15:34:41 | metatlas2.ms1_ms2_analysis | INFO |   Hit detection 3/14 complete: CKLJMWTZIZZHCS-NYTNKSBQSA-N
2025-09-16 15:34:41 | metatlas2.ms1_ms2_analysis | INFO |   Hit detection 4/14 complete: WHUUTDBJXJRKMK-UFLWJPECSA-N
2025-09-16 15:34:41 | metatlas2.ms1_ms2_analysis | INFO |   Hit detection 5/14 complete: UYTPUPDQBNUYGX-CIKZIQIKSA-N
2025-09-16 15:34:41 | metatlas2.ms1_ms2_analysis | INFO |   Hit detection 6/14 complete: FDGQSTZJBFJUBT-NNZQUYKOSA-N
2025-09-16 15:34:41 | metatlas2.ms1_ms2_analysis | INFO |   Hit detection 7/14 complete: UGQMRVRMYYASKQ-SHKQESLVSA-N
2025-09-16 15:34:41 | metatlas2.ms1_ms2_analysis | INFO |   Hit detection 8/14 complete: KZSNJWFQEVHDMF-XAFSXMPTSA-N
2025-09-16 15:34:41 | metatlas2.ms1_ms2_analysis | INFO |   Hit 

Processing files in parallel:   0%|          | 0/6 [00:00<?, ?it/s]

2025-09-16 15:34:45 | metatlas2.ms1_ms2_analysis | INFO |   Completed 1/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_22_Fall-RFa_7_Rg70to1050-CE205060norm-rhizo-InjBL-MeOH_Run123.h5 - 35 compounds
2025-09-16 15:34:45 | metatlas2.ms1_ms2_analysis | INFO |   Completed 2/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_FPS_MS1_28_Spring-VSp_2_Rg70to1050-CE102040norm-veg-core-ISTD_Run31.h5 - 49 compounds
2025-09-16 15:34:46 | metatlas2.ms1_ms2_analysis | INFO |   Completed 3/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_36_Fall-VFa_7_Rg70to1050-CE102040norm-veg-core-S1_Run136.h5 - 53 compounds
2025-09-16 15:34:46 | metatlas2.ms1_ms2_analysis | INFO |   Completed 4/6: 20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519_POS_MS2_48_Fall-BFa_7_Rg70to1050-CE102040norm-nonveg-core-S1_Run127.h5 - 56 compounds
2025-09-16 15:34:46 | metatlas2.ms1_ms2_analysis | INFO |   Completed 5/6: 20250707_JGI_MS_510172

Finding MS2 hits in parallel:   0%|          | 0/36 [00:00<?, ?it/s]

2025-09-16 15:34:52 | metatlas2.ms1_ms2_analysis | INFO |   Hit detection 1/36 complete: OVRNDRQMDRJTHS-ZEUBEQSHSA-N
2025-09-16 15:34:52 | metatlas2.ms1_ms2_analysis | INFO |   Hit detection 2/36 complete: GFFGJBXGBJISGV-CIKZIQIKSA-N
2025-09-16 15:34:53 | metatlas2.ms1_ms2_analysis | INFO |   Hit detection 3/36 complete: GFFGJBXGBJISGV-UHFFFAOYSA-N
2025-09-16 15:34:53 | metatlas2.ms1_ms2_analysis | INFO |   Hit detection 4/36 complete: QNAYBMKLOCPYGJ-REOHCLBHSA-N
2025-09-16 15:34:53 | metatlas2.ms1_ms2_analysis | INFO |   Hit detection 5/36 complete: CKLJMWTZIZZHCS-NYTNKSBQSA-N
2025-09-16 15:34:53 | metatlas2.ms1_ms2_analysis | INFO |   Hit detection 6/36 complete: OPTASPLRGRRNAP-UHFFFAOYSA-N
2025-09-16 15:34:53 | metatlas2.ms1_ms2_analysis | INFO |   Hit detection 7/36 complete: WHUUTDBJXJRKMK-UFLWJPECSA-N
2025-09-16 15:34:53 | metatlas2.ms1_ms2_analysis | INFO |   Hit detection 8/36 complete: WHUUTDBJXJRKMK-VKHMYHEASA-N
2025-09-16 15:34:53 | metatlas2.ms1_ms2_analysis | INFO |   Hit 

## Schema Overview

This workflow implements the following schema:

### 1. Adding Compounds to Database:
```
DatabaseManager (organizer)
    ├── Compound (read from input file + PubChem queries → compounds table)
    └── CompoundReference (read from input file → mz_rt_references table, skipped if RT/MZ missing)
```

### 2. Adding Atlases to Database:
```
AtlasManager (organizer)
    └── Atlas (new atlas with new UID → atlas table)
        └── Compound (inchi_keys from input file, must exist in compounds table)
            └── CompoundReference (reuse if exact match exists, otherwise create new)
```

### 3. Running Targeted Analysis:
```
TargetedAnalysisManager (organizer)
    └── WorkflowResults (results container)
        └── Atlas (loaded by UID from database)
            └── Compound (comes with atlas)
                ├── CompoundReference (comes with atlas)
                └── CompoundExperimental (created during analysis → mz_rt_experimental table)
```